# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități masive de text și cod pentru a înțelege, genera și manipula limbajul uman. Aceste modele pot realiza o gamă largă de sarcini lingvistice, cum ar fi răspunsul la întrebări, scrierea de texte creative și traducerea.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități masive de text și cod, capabilă să înțeleagă și să genereze limbaj uman într-un mod fluent și coerent. Această capacitate îi permite să efectueze o varietate de sarcini, cum ar fi răspunsul la întrebări, traducerea limbilor, scrierea de texte creative și chiar rezumarea informațiilor complexe.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, politica românească a fost marcată de o instabilitate guvernamentală frecventă, cu multiple schimbări de premier și coaliții. De asemenea, s-a observat o accentuare a luptei anticorupție și o consolidare a rolului instituțiilor judiciare în viața publică.

--- Gemini 2.5 Flash ---
Principala schimbare a fost formarea unei coaliții guvernamentale fără precedent între partidele tradițional rivale PNL și PSD. Aceasta a condus la o perioadă de stabilitate guvernamentală, marcată de o rotație a premierilor între cele două partide.

--- OpenRouter Free ---
În ultimii 5 ani, principalul schimbament politic a fost victoria lui Klaus Iohannis în alegerile prezidențiale din 2019, asigurându-i un al doilea mandat. Al doilea schimbament major a fost revenirea Partidului Social Democrat (PSD) ca cel mai mare partid în Parlament după alegerile legislative din 2020.


In [7]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd() / ".env"

print("Current folder:", Path.cwd())
print(".env exists here:", env_path.exists())

load_dotenv(env_path)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))

Current folder: c:\Users\valer\Desktop\Analiza Datelor Complexe\17. AI Avansat\echochamber-project-team3\notebooks\student_01
.env exists here: False
OpenRouter key loaded: True


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [8]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, cinic
Emoție dominantă: Frustrare, neîncredere
Țintă principală: Clasa politică
Populism: da

--- Gemini 2.5 Flash ---
Ton: Acuzator, cinic.
Emoție dominantă: Frustrare, neîncredere.
Țintă principală: Clasa politică.
Populism: da

--- OpenRouter Free ---
Ton: Generalizator, acuzator, dezamăgit.
Emoție dominantă: Furie și frustrare față de clasa politică.
Țintă principală: Întreaga clasă politică, prezentată ca unitate coruptă și indiferentă.
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [9]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [15]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Returnează DOAR JSON valid.
Nu folosi markdown.
Nu adăuga explicații în afara JSON-ului.
"""

PROMPT = f"""
Adnotează următorul comentariu politic:
"{COMENTARIU}"

Returnează strict un JSON cu exact aceste chei:
{{
  "ton": "pozitiv/negativ/neutru",
  "emotie_dominanta": "furie/frica/speranta/dezamagire/ironie/neutru",
  "tinta_principala": "text scurt",
  "populism": true/false,
  "explicatie_scurta": "maxim 8 cuvinte"
}}
"""

In [17]:
import json
import re

def extrage_json(text):
    text = str(text).strip()

    start = text.find("{")
    end = text.rfind("}") + 1

    if start == -1 or end == 0:
        raise ValueError("Nu am găsit JSON în răspuns:\n" + text)

    json_text = text[start:end]
    return json.loads(json_text)


for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raw = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    )

    try:
        rezultat = extrage_json(raw)
        print(rezultat)
    except Exception as e:
        print("[Eroare JSON]", e)
        print("RAW RESPONSE:")
        print(raw)


--- Gemini 2.5 Flash Lite ---
[Eroare JSON] Nu am găsit JSON în răspuns:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]
RAW RESPONSE:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare JSON] Nu am găsit JSON în răspuns:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]
RAW RESPONSE:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'Politicieni corupți, publicul suferă', 'populism': True, 'explicatie_scurta': 'Critică la adresa corupției politice'}


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [18]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

temperature=0.7:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

temperature=1.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

[ Gemini 2.5 Flash ]

temperature=0.1:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

temperature=0.7:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

temperature=1.2:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

[ OpenRouter Free ]

temperature=0.1:
Această situație înseamnă că alegerile au fost decisă de un comit constitucional, ceea ce implica că toate partidele sau grupurile politice au avut o posiție în discuție.

Această informație poate ajuta la comprima că procesul de votare este complex și influențat de opinii diverse, în mod general.

temperature=0.7:
Anularea alegerilor de către Curtea Constituționa

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da | da | da | Răspunde bine, dar a avut quota/rate limit. |
| OpenRouter Free | parțial | parțial | parțial | da | A răspuns la texte simple, dar JSON-ul a fost instabil. |
| Gemini 2.5 Flash | da | da | da | da | Cel mai bun răspuns, dar poate avea limită de quota. |
### Decizie
**Model principal ales:Gemini 2.5 Flash Lite
**Model de rezervă:OpenRouter Free
**Temperature recomandată: 0.1
**De ce am ales acest model?**  
Am ales Gemini 2.5 Flash ca model principal deoarece a răspuns bine în limba română și a respectat instrucțiunile pentru adnotarea comentariilor politice. Modelul a identificat corect tonul, emoția dominantă, ținta principală și caracterul populist al comentariilor, deci poate fi folosit pentru adnotare. Totuși, stabilitatea este afectată de limitele de quota/rate limit, de aceea OpenRouter Free poate fi folosit ca rezervă, dar cu verificare suplimentară pentru output JSON.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [19]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales